# 02 - Cleaning and data quality

We apply `clean_games` (in `src/cleaning.py`) and document what each filter removes.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.data_loader import load_games
from src.cleaning import RAW_TO_CLEAN, clean_games, parse_date, parse_owners, clean_tag_string, load_clean

## 1. Test the small functions by hand

In [2]:
print(parse_date('Aug 1, 2023'))
print(parse_date('not a date'))         # None, without crashing
print(parse_owners('20000 - 50000'))
print(parse_owners(None))               # (None, None), without crashing
print(clean_tag_string(' Indie , Roguelike,, Action '))

2023-08-01 00:00:00
None
(20000, 50000)
(None, None)
Indie,Roguelike,Action


## 2. Load and clean

In [3]:
raw = load_games('../data/games.csv', usecols=list(RAW_TO_CLEAN.keys()))
df, report = clean_games(raw)
df.shape

(71352, 10)

## 3. Quality report

In [4]:
report

,step,rows removed,% of raw removed,rows left,reason
0,Start (raw file),0,0.0,125855,
1,Contract columns,0,0.0,125855,"Keep the useful columns, renamed in snake_case"
2,Duplicates and unnamed,1,0.0,125854,A game counted twice distorts the statistics
3,100 % discount,6,0.0,125848,Full price cannot be recovered (other discount...
4,Free games,26656,21.2,99192,"No price, so no sales revenue to estimate"
5,Unreadable dates,0,0.0,99192,The game cannot be placed in time
6,Unreadable owners,0,0.0,99192,The sales estimate cannot be checked
7,Games without tags,24478,19.4,74714,They cannot be compared to a concept
8,Released less than 6 months ago,247,0.2,74467,Their reviews have not had time to come in
9,Released before 2015,3115,2.5,71352,The pre-2015 market does not look like today's


In [5]:
# Check: starting rows - removed rows = final rows
start = report['rows left'].iloc[0]
removed = report['rows removed'].sum()
print(start, '-', removed, '=', start - removed, '| final rows:', len(df))
assert start - removed == len(df)

125855 - 54503 = 71352 | final rows: 71352


The two biggest filters are free games (26,656) and games without tags (24,478).
In the end, we keep 71,352 paid, tagged games released between 2015 and August 2025.

**Trap no. 2**: the `Price` column is the price on the day of collection, **discount included** (41 % of paid games were on sale).
`restore_full_price` recovers the full price: price / (1 - discount / 100). The 6 games at -100 % are removed (full price cannot be recovered).

In [6]:
raw_hk = raw[raw['Name'] == 'Hollow Knight'][['Name', 'Price', 'Discount']]
clean_hk = df[df['name'] == 'Hollow Knight'][['name', 'price']]
print(raw_hk.to_string(index=False))
print(clean_hk.to_string(index=False))

         Name  Price  Discount
Hollow Knight   7.49        50
         name  price
Hollow Knight  14.98


Hollow Knight: $7.49 at -50 % in the raw file, $14.98 full price after the fix ($14.99 in reality: the gap comes from the rounding of the discounted price).
**Why 2015**: before that, Steam was a curated catalogue (fewer than 2,500 releases a year); the market does not look like today's.

## 4. strptime versus pd.to_datetime

In [7]:
dates_strptime = pd.to_datetime(raw['Release date'].apply(parse_date))
dates_pandas = pd.to_datetime(raw['Release date'], format='%b %d, %Y', errors='coerce')
print('Identical results:', (dates_strptime == dates_pandas).all())

Identical results: True


Both methods give the same result. `pd.to_datetime` is faster; the `strptime` + `try/except` version shows explicitly what happens to unreadable dates.

## 5. Save, reload, check

In [8]:
df.to_csv('../data/games_clean.csv', index=False)
clean = load_clean('../data/games_clean.csv')
clean.dtypes

app_id                   int64
name                       str
release_date    datetime64[us]
release_year             int64
price                  float64
positive                 int64
negative                 int64
owners_min               int64
owners_max               int64
tags                       str
dtype: object

In [9]:
print('Prices at 0:', (clean['price'] == 0).sum())
print('Years:', clean['release_year'].min(), 'to', clean['release_year'].max())
print('owners_min <= owners_max everywhere:', (clean['owners_min'] <= clean['owners_max']).all())
clean.head()

Prices at 0: 0
Years: 2015 to 2025
owners_min <= owners_max everywhere: True


,app_id,name,release_date,release_year,price,positive,negative,owners_min,owners_max,tags
0,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,2016,14.97,252,3,0,20000,"Adventure,Visual Novel,Anime,Cute"
1,1034400,Mystery Solitaire The Black Raven,2019-05-06,2019,4.99,21,3,0,20000,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."
2,1934300,Armored Brigade II,2025-04-08,2025,39.99,117,13,0,20000,"Simulation,Strategy,RTS,Wargame,Grand Strategy..."
3,1157670,Hepta Beats,2021-05-07,2021,0.99,28,4,0,20000,"Precision Platformer,Rhythm,Music,Difficult,Ab..."
4,1540330,MUMBA IV: Egypt Jewels,2021-12-13,2021,0.59,19,6,0,20000,"Casual,Action,Strategy,Puzzle,Match 3,Singlepl..."
